In [ ]:
from __future__ import annotations

import gc
import os
from datetime import datetime
from pathlib import Path
from typing import Final

import polars as pl
import pyarrow.parquet as pq

now = datetime.now()

print(f"Last notebook execution: {now}")

Last notebook execution: 2026-11-05 21:38:03.929375


In [2]:
size_gb = os.path.getsize("../tables/all_predictions.parquet") / 1e9
print(f"Parquet size: {size_gb:.1f} GB")

Parquet size: 2.9 GB


In [ ]:
"""One-time backfill: add model/seed/ctx/run/subject_id columns, save as parquet."""


results_dir = Path("../results")
csv_paths = [
    p
    for p in sorted(results_dir.rglob("predictions.csv"))
    if len(p.parent.relative_to(results_dir).parts) >= 4
]
print(f"Found {len(csv_paths)} files")

out_path = Path("../tables/all_predictions.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)

writer = None
total_rows = 0

for csv_path in csv_paths:
    parts = csv_path.parent.relative_to(results_dir).parts
    df = pl.read_csv(csv_path, separator=";")
    df = df.with_columns(
        [
            pl.col("y_true").cast(pl.Float64),
            pl.col("y_pred").cast(pl.Float64),
            pl.col("y_proba_pos").cast(pl.Float64),
        ]
    )
    df = df.with_columns(
        [
            pl.lit(parts[2]).alias("model"),
            pl.lit(parts[1]).alias("run"),
            pl.lit(int(parts[3].split("_")[1])).alias("seed"),
            pl.lit(int(parts[3].split("_ctx_")[1])).alias("ctx"),
        ]
    )
    df = df.with_columns(pl.col("path").str.split("/").list.get(-3).alias("subject_id"))

    arrow_table = df.to_arrow()
    if writer is None:
        writer = pq.ParquetWriter(out_path, arrow_table.schema, compression="zstd")
    writer.write_table(arrow_table)
    total_rows += df.height

    del df, arrow_table
    gc.collect()

writer.close()
print(f"Saved {total_rows:,} rows -> {out_path}")
del writer
gc.collect()

Found 300 files
Saved 40,589,298 rows -> ../tables/all_predictions.parquet


0

In [3]:
lf = pl.scan_parquet("../tables/all_predictions.parquet")
print(lf.collect_schema())
print(f"Rows: {lf.select(pl.len()).collect().item():,}")

Schema({'path': String, 'y_true': Float64, 'y_pred': Float64, 'y_proba_pos': Float64, 'gaze_yaw': Float64, 'gaze_pitch': Float64, 'au01': Float64, 'au02': Float64, 'au03': Float64, 'au04': Float64, 'au05': Float64, 'au06': Float64, 'au07': Float64, 'au08': Float64, 'model': String, 'run': String, 'seed': Int32, 'ctx': Int32, 'subject_id': String})
Rows: 40,589,298


In [4]:
lf.columns

/tmp/ipykernel_640279/3701027225.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  lf.columns


['path',
 'y_true',
 'y_pred',
 'y_proba_pos',
 'gaze_yaw',
 'gaze_pitch',
 'au01',
 'au02',
 'au03',
 'au04',
 'au05',
 'au06',
 'au07',
 'au08',
 'model',
 'run',
 'seed',
 'ctx',
 'subject_id']

In [ ]:
"""Generate TFM ensemble predictions by averaging y_proba_pos across models."""


INPUT_PATH: Final[str] = "../tables/all_predictions.parquet"
ENSEMBLE_PATH: Final[Path] = Path("../tables/ensemble_predictions.parquet")
ENSEMBLE_NAME: Final[str] = "ensemble"
TFM_MODELS: Final[list[str]] = ["tabpfn", "tabicl", "tabdpt", "causilo", "kumo"]
N_TFMS: Final[int] = len(TFM_MODELS)

lf = pl.scan_parquet(INPUT_PATH)

# Filter to TFM models only
tfm_lf = lf.filter(pl.col("model").is_in(TFM_MODELS))

# Count distinct models per sample — keep only groups where ALL TFMs are present
sample_counts = tfm_lf.group_by(["path", "seed", "ctx"]).agg(
    pl.col("model").n_unique().alias("n_models")
)

complete_samples = sample_counts.filter(pl.col("n_models") == N_TFMS)

# Aggregate: mean probability, take first for identical columns
ensemble_lf = (
    tfm_lf.join(complete_samples, on=["path", "seed", "ctx"], how="inner")
    .group_by(["path", "seed", "ctx", "y_true", "subject_id"])
    .agg(
        pl.col("y_proba_pos").mean().alias("y_proba_pos"),
        pl.col("gaze_yaw").first().alias("gaze_yaw"),
        pl.col("gaze_pitch").first().alias("gaze_pitch"),
        pl.col("au01").first().alias("au01"),
        pl.col("au02").first().alias("au02"),
        pl.col("au03").first().alias("au03"),
        pl.col("au04").first().alias("au04"),
        pl.col("au05").first().alias("au05"),
        pl.col("au06").first().alias("au06"),
        pl.col("au07").first().alias("au07"),
        pl.col("au08").first().alias("au08"),
        pl.col("run").first().alias("run"),
    )
    .with_columns(
        (pl.col("y_proba_pos") >= 0.5).cast(pl.Float64).alias("y_pred"),
        pl.lit(ENSEMBLE_NAME).alias("model"),
    )
)

# Stream to disk — polars 2.1+ spill-to-disk handles join + group_by
ENSEMBLE_PATH.parent.mkdir(parents=True, exist_ok=True)
ensemble_lf.sink_parquet(ENSEMBLE_PATH)

# Row count from output metadata (negligible memory)
n_ensemble = pl.scan_parquet(ENSEMBLE_PATH).select(pl.len()).collect().item()
print(f"Saved {n_ensemble:,} ensemble rows -> {ENSEMBLE_PATH}")

# Free resources
del lf, tfm_lf, sample_counts, complete_samples, ensemble_lf
gc.collect()
print("Memory freed.")

Saved 6,764,646 ensemble rows -> ../tables/ensemble_predictions.parquet
Memory freed.
